# Benchmark results

Tables from the outputs of `run_benchmark.py` (see `benchmark.py` for the protocol). To reproduce:

```
python run_benchmark.py impute --horizon 6
python run_benchmark.py fit --horizon 6 --model all
python run_benchmark.py impute --horizon 8
python run_benchmark.py fit --horizon 8 --model all
```

Each model's outputs are in `results/h{6,8}/{model}/`: `metrics.csv`, `selection.json`, `validation_grid.csv`, and the per-person predictions (not published).
Populations: `all` = every participant (missing predictors imputed by the training-only imputer; primary), `complete` = participants with complete predictors (sensitivity).

In [ ]:
import glob
import json
import os

import pandas as pd

MODELS = ['LR', 'GNB', 'DT', 'LightGBM', 'CatBoost', 'RealMLP', 'TabICL', 'GAT']
METRICS = ['AUROC', 'AUPRC', 'Precision', 'Recall', 'F1', 'Accuracy', 'Brier']

frames = []
for path in glob.glob('results/h*/*/metrics.csv'):
    horizon, model = path.split(os.sep)[-3:-1]
    frames.append(pd.read_csv(path).assign(horizon=int(horizon[1:]), model=model))
metrics = pd.concat(frames, ignore_index=True)
metrics['model'] = pd.Categorical(metrics['model'], MODELS, ordered=True)
print(metrics.groupby(['horizon', 'model'], observed=True).size())

## Internal validation: mean (SD) over 10 repeats

In [ ]:
def mean_sd(df):
    g = df.groupby(['horizon', 'population', 'model'], observed=True)[METRICS]
    m, s = g.mean() * 100, g.std() * 100
    out = m.round(1).astype(str) + ' (' + s.round(1).astype(str) + ')'
    out['Brier'] = (m['Brier'] / 100).round(3).astype(str) + ' (' + (s['Brier'] / 100).round(3).astype(str) + ')'
    return out

internal = mean_sd(metrics[metrics['setting'] == 'internal'])
internal

## Temporal validation (2010 cohort)

In [ ]:
temporal = metrics[metrics['setting'] == 'temporal'].set_index(['horizon', 'population', 'model'])
temporal_table = temporal[METRICS].copy()
temporal_table[METRICS[:-1]] = (temporal_table[METRICS[:-1]] * 100).round(1)
temporal_table['Brier'] = temporal_table['Brier'].round(3)
temporal_table[['n', 'cases']] = temporal[['n', 'cases']]
temporal_table.sort_index()

## Selected hyperparameters and thresholds

In [ ]:
rows = []
for path in glob.glob('results/h*/*/selection.json'):
    sel = json.load(open(path))
    picks = pd.Series([str(r['params']) for r in sel['per_repeat']]).value_counts()
    rows.append({'horizon': sel['horizon'], 'model': sel['model'],
                 'temporal configuration': str(sel['temporal']['params']),
                 'temporal threshold': round(sel['temporal']['threshold'], 3),
                 'internal selections (count of 10)': '; '.join(f'{k}: {v}' for k, v in picks.items())})
pd.DataFrame(rows).sort_values(['horizon', 'model'])

## Temporal validation by overlap with the development cohort

HRS follows the same people and households, so part of the 2010 cohort was in the 2000 development cohort. Performance is shown separately for returning participants, new participants from a development household, and new participants from new households (independent of the development data at both the participant and household level). Primary population (`all`).

In [ ]:
import benchmark
from cohort_tables import overlap_status
from sklearn.metrics import average_precision_score, roc_auc_score

rows = []
for horizon in (6, 8):
    dev, ext, P = benchmark.load_cohorts(horizon)
    for model in MODELS:
        path = f'results/h{horizon}/{model}/temporal_predictions.csv'
        if not os.path.exists(path):
            continue
        pred = pd.read_csv(path)
        pred['overlap'] = overlap_status(pred['hhidpn'], dev['hhidpn'])
        for group, d in [('all', pred)] + list(pred.groupby('overlap')):
            rows.append({'horizon': horizon, 'model': model, 'group': group, 'n': len(d), 'cases': int(d['y'].sum()),
                         'AUROC': round(roc_auc_score(d['y'], d['p']) * 100, 1),
                         'AUPRC': round(average_precision_score(d['y'], d['p']) * 100, 1)})
overlap = pd.DataFrame(rows)
overlap.pivot_table(index=['horizon', 'model'], columns='group', values=['AUROC', 'AUPRC'])

## Cohort tables (from `cohort_tables.py`)

In [ ]:
for name in ['participant_flow', 'complete_vs_incomplete', 'cohort_overlap']:
    display(pd.read_csv(f'results/tables/{name}.csv'))